# Infra-FM: STAC Imagery Fetch

Fetches Sentinel-2 + Sentinel-1 tiles for infrastructure assets across multiple regions.

**Regions in this notebook:** australia-oceania, africa, south-america

**Before running:**
1. Upload your deduped parquet files to Google Drive:
   - `infra_fm/pipeline/africa_deduped_assets_substations.parquet`
   - `infra_fm/pipeline/australia-oceania_deduped_assets_substations.parquet`
   - `infra_fm/pipeline/south-america_deduped_assets_substations.parquet`
2. Upload your curation code zip to Drive: `infra_fm/code/infra_fm_curation.zip`
3. Run cells top to bottom — checkpoints save to Drive so you can resume if disconnected.

## 1. Mount Google Drive

In [ ]:
import psutil
import subprocess

ram = psutil.virtual_memory()
print(f'RAM: {ram.available/1e9:.1f}GB available / {ram.total/1e9:.1f}GB total')
print(f'Used: {ram.used/1e9:.1f}GB ({ram.percent:.0f}%)')

# check top memory consumers
result = subprocess.run(['ps', 'aux', '--sort=-%mem'], 
                       capture_output=True, text=True)
lines = result.stdout.split('\n')
print('\nTop memory consumers:')
for line in lines[:10]:
    print(line)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# verify your files are visible
import os
DRIVE_ROOT = '/content/drive/MyDrive/infra_fm'
print('Drive root contents:')
for f in sorted(os.listdir(DRIVE_ROOT)):
    print(' ', f)

## 2. Install dependencies

In [3]:
!pip install -q pystac-client planetary-computer rasterio scipy opencv-python-headless

## 3. Set up curation code

In [ ]:
import os, sys, zipfile
from pathlib import Path

ZIP_PATH   = '/content/drive/MyDrive/infra_fm/code/infra_fm_curation.zip'
EXTRACT_TO = '/content/infrabench_repo'

# curation is a package whose modules import each other relatively, so the
# whole tree has to be extracted intact and only the repo root goes on
# sys.path. extracting loose modules into /content no longer works.
os.makedirs(EXTRACT_TO, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    for member in z.namelist():
        clean  = member.replace(chr(92), '/')
        target = os.path.join(EXTRACT_TO, clean)
        if clean.endswith('/'):
            os.makedirs(target, exist_ok=True)
        else:
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(member) as src, open(target, 'wb') as dst:
                dst.write(src.read())

# the zip may or may not carry a top-level folder, so probe both layouts
REPO_ROOT = next(
    (c for c in [EXTRACT_TO, f'{EXTRACT_TO}/infra_fm_code_only']
     if Path(f'{c}/curation/__init__.py').exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(f'no curation package found under {EXTRACT_TO}')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
print(f'repo root: {REPO_ROOT}')


In [ ]:
from curation.stac_imagery   import STACImageryFetcher
from curation.qc             import QualityChecker
from curation.triage         import RuleBasedTriager
from curation.dataset        import DatasetAssembler
from curation.utils.io_utils import load_asset_table

print('curation imports OK')


In [ ]:
import psutil
import subprocess

ram = psutil.virtual_memory()
print(f'RAM: {ram.available/1e9:.1f}GB available / {ram.total/1e9:.1f}GB total')
print(f'Used: {ram.used/1e9:.1f}GB ({ram.percent:.0f}%)')

# check top memory consumers
result = subprocess.run(['ps', 'aux', '--sort=-%mem'], 
                       capture_output=True, text=True)
lines = result.stdout.split('\n')
print('\nTop memory consumers:')
for line in lines[:10]:
    print(line)

## 4. Configuration — edit this cell before running

In [ ]:
# --- paths ---
PIPELINE_DIR   = f'{DRIVE_ROOT}/pipeline'       # where your deduped parquets live
DATASETS_DIR   = f'{DRIVE_ROOT}/datasets'       # where curated datasets will be written
CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints'    # fetch checkpoints — survives disconnects

os.makedirs(DATASETS_DIR,   exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# --- fetch settings ---
MODALITIES   = ['sentinel2_ms', 'sentinel1']   # landsat_thermal dropped for speed
BUFFER_M     = 300
MAX_WORKERS  = 16    # Colab has good network — push concurrency higher than local
START_WORKERS = 8

# --- regions to process (in order — smallest first) ---
REGIONS = [
    'europe'
]

# skip regions that already have a _SUCCESS file
SKIP_DONE = True

print('Configuration:')
print(f'  Modalities:    {MODALITIES}')
print(f'  Buffer:        {BUFFER_M}m')
print(f'  Workers:       {START_WORKERS} start / {MAX_WORKERS} max')
print(f'  Regions:       {REGIONS}')
print(f'  Datasets dir:  {DATASETS_DIR}')
print(f'  Checkpoints:   {CHECKPOINT_DIR}')

## Debug

In [ ]:
import psutil, os

# overall RAM
ram = psutil.virtual_memory()
print(f'RAM: {ram.available/1e9:.1f}GB available / {ram.total/1e9:.1f}GB total')
print(f'Used: {ram.used/1e9:.1f}GB ({ram.percent:.0f}%)')

# check if the infra_fm folder from the old extraction is eating space
import shutil
for folder in ['/content/infra_fm']:
    if os.path.exists(folder):
        size = shutil.disk_usage(folder).used / 1e9
        print(f'{folder}: {size:.2f} GB on disk')

## 5. Verify deduped parquets are accessible

In [ ]:
from pathlib import Path
import pandas as pd

for region in REGIONS:
    path = f'{PIPELINE_DIR}/{region}_deduped_assets_substations_sampled.parquet'
    if Path(path).exists():
        # read only first row to verify file exists — don't load full dataframe
        df_peek = pd.read_parquet(path, columns=['asset_id'])
        print(f'{region:25s} {len(df_peek):>6,} assets — OK')
        del df_peek  # immediately free memory
    else:
        print(f'{region:25s} MISSING — {path}')

In [ ]:
import psutil
import subprocess

ram = psutil.virtual_memory()
print(f'RAM: {ram.available/1e9:.1f}GB available / {ram.total/1e9:.1f}GB total')
print(f'Used: {ram.used/1e9:.1f}GB ({ram.percent:.0f}%)')

# check top memory consumers
result = subprocess.run(['ps', 'aux', '--sort=-%mem'], 
                       capture_output=True, text=True)
lines = result.stdout.split('\n')
print('\nTop memory consumers:')
for line in lines[:10]:
    print(line)

## 6. Run STAC fetch pipeline — all regions

This cell runs the full pipeline (fetch → QC → triage → assemble) for each region.
Checkpoints are saved to Drive every 200 tiles — if the session disconnects, re-run
this cell and it will resume from the last checkpoint automatically.

In [ ]:
import os
import json
from datetime import datetime

import psutil

from curation.stac_imagery   import STACImageryFetcher
from curation.qc             import QualityChecker
from curation.triage         import RuleBasedTriager
from curation.dataset        import DatasetAssembler
from curation.utils.io_utils import load_asset_table

ram = psutil.virtual_memory()
print(f'RAM available: {ram.available / 1e9:.1f} GB / {ram.total / 1e9:.1f} GB total')


def is_done(region):
    success = Path(f'{DATASETS_DIR}/dataset_{region}_stac_v1/_SUCCESS')
    return success.exists()


def run_region(region):
    print('\n' + '=' * 60)
    print(f'Region: {region}  ({datetime.now().strftime("%H:%M:%S")})')
    print('=' * 60)

    if SKIP_DONE and is_done(region):
        print(f'  Already complete — skipping.')
        return

    if region == 'europe':
        parquet = f'{PIPELINE_DIR}/europe_deduped_assets_substations_sampled.parquet'
    else:
        parquet = f'{PIPELINE_DIR}/{region}_deduped_assets_substations.parquet'
    
    df = load_asset_table(parquet)
    print(f'  Loaded {len(df):,} assets')
    del df  # free memory before fetch

    # reload for fetcher
    df = load_asset_table(parquet)
    
    output_dir      = f'{DATASETS_DIR}/dataset_{region}_stac_v1'
    checkpoint_path = f'{CHECKPOINT_DIR}/{region}_fetch.pkl'
    os.makedirs(output_dir, exist_ok=True)

    print(f'  [1/4] Fetching imagery...')
    try:
        fetcher = STACImageryFetcher(
            buffer_m             = BUFFER_M,
            modalities           = MODALITIES,
            temporal_stack       = False,
            checkpoint_path      = checkpoint_path,
            checkpoint_every     = 200,
            adaptive_concurrency = True,
            start_workers        = 4,
            max_workers          = 16,
        )
        print('  Fetcher created OK')
    except Exception as e:
        print(f'  FETCHER CREATION FAILED: {e}')
        import traceback
        traceback.print_exc()
        return

# =============================================================
    import psutil
    ram = psutil.virtual_memory()
    print(f'RAM before fetch: {ram.available/1e9:.1f}GB available')
# =============================================================

    tiles = fetcher.fetch_all(df)
    n_ok  = sum(1 for t in tiles if t.status == 'ok')
    print(f'  Fetched: {n_ok} ok / {len(tiles) - n_ok} failed')

    # --- step 2: QC ---
    print(f'  [2/4] Quality control...')
    checker    = QualityChecker(min_valid_ratio=0.80)
    qc_results = checker.check_all(tiles, max_workers=4)
    clean      = checker.filter_ok(qc_results)
    print(f'  QC passed: {len(clean)} / {len(tiles)}')

    # --- step 3: Triage ---
    print(f'  [3/4] Confidence triage...')
    triager        = RuleBasedTriager(contradiction_threshold=3, low_threshold=4)
    triage_results = triager.triage_all(clean, max_workers=4)
    accepted       = triager.filter_accepted(triage_results)
    print(f'  Accepted: {len(accepted)}')

    # --- step 4: Assemble ---
    print(f'  [4/4] Assembling dataset -> {output_dir}')
    assembler = DatasetAssembler(output_dir)
    summary   = assembler.assemble(accepted, triage_results)

    # write _SUCCESS
    success_path = Path(output_dir) / '_SUCCESS'
    with open(success_path, 'w') as f:
        json.dump({
            'completed_at':   datetime.utcnow().isoformat() + 'Z',
            'region':         region,
            'n_dataset_tiles': len(summary),
            'modalities':     MODALITIES,
        }, f, indent=2)

    print(f'  Done. {len(summary)} tiles assembled.')
    return len(summary)


# run all regions
results = {}
for region in REGIONS:
    try:
        n = run_region(region)
        results[region] = n or 'skipped'
    except Exception as e:
        print(f'ERROR in {region}: {e}')
        results[region] = f'error: {e}'

print('\n' + '=' * 40)
print('SUMMARY')
print('=' * 40)
for region, result in results.items():
    print(f'  {region:25s} {result}')

## 7. Verify completed datasets

In [ ]:
import json

print('Dataset status:')
for region in REGIONS:
    success_path = Path(f'{DATASETS_DIR}/dataset_{region}_stac_v1/_SUCCESS')
    if success_path.exists():
        meta = json.loads(success_path.read_text())
        print(f'  {region:25s} DONE — {meta["n_dataset_tiles"]:,} tiles')
    else:
        # check if partially complete via checkpoint
        ckpt = Path(f'{CHECKPOINT_DIR}/{region}_fetch.pkl')
        if ckpt.exists():
            import pickle
            data = pickle.load(open(ckpt, 'rb'))
            n_done = len(data.get('completed_ids', []))
            print(f'  {region:25s} IN PROGRESS — {n_done:,} tiles fetched so far')
        else:
            print(f'  {region:25s} NOT STARTED')

## 8. Download completed datasets to Drive (already done — they write there directly)

Your datasets are written directly to `My Drive/infra_fm/datasets/`.
To use them locally:
1. Download each `dataset_<region>_stac_v1/` folder from Drive to your local `data/curated_datasets/`
2. Or run pretraining directly from Colab (see pretraining notebook)

In [ ]:
import os

print("Contents of /content/ (top level):")
for item in sorted(os.listdir('/content')):
    if not item.startswith('.') and item not in ['drive', 'sample_data']:
        print(f"  {item}")

print("\nContents of /content/legacy/ (if exists):")
if os.path.exists('/content/legacy'):
    for item in sorted(os.listdir('/content/legacy')):
        print(f"  {item}")
else:
    print("  (does not exist)")

print("\nContents of /content/helpers/ (if exists):")
if os.path.exists('/content/helpers'):
    for item in sorted(os.listdir('/content/helpers')):
        print(f"  {item}")
else:
    print("  (does not exist)")

print("\nContents of /content/utils/ (if exists):")
if os.path.exists('/content/utils'):
    for item in sorted(os.listdir('/content/utils')):
        print(f"  {item}")
else:
    print("  (does not exist)")

print("\nKey files present:")
for f in ['stac_imagery.py', 'qc.py', 'triage.py', 'dataset.py', 'sources.py']:
    print(f"  {f}: {os.path.exists(f'/content/{f}')}")